# 04 Python Unit Tests - pytest - fixtures
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **Fixtures - definicja i wstrzykiwanie zaleznosci** - `@pytest.fixture`
2. 🔹 **Scope i cykl zycia** - `function`, `class`, `module`, `session`, `params`
3. 🔹 **Yield fixtures i `conftest.py`** - teardown, wspoldzielenie
4. 🔹 **Wbudowane fixtures** - `tmp_path`, `capsys`, `monkeypatch`

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 Fixtures - definicja i wstrzykiwanie zaleznosci

Fixture (armatura testowa) to mechanizm przygotowywania danych
i zasobow dla testow w pytest. Zastepuje `setUp()` z `unittest`,
ale jest bardziej elastyczna - mozna ja uzywac selektywnie,
laczyc z innymi fixtures i wspoldzielic miedzy testami.

**Definicja:** dekorator `@pytest.fixture` przed funkcja:

```python
@pytest.fixture
def bank_account():
    return BankAccount(balance=100)
```

**Wstrzykiwanie:** wystarczy podac nazwe fixture jako parametr
funkcji testowej - pytest wstrzykuje wartosc automatycznie:

```python
def test_deposit(bank_account):   # <- pytest wstrzykuje tu
    bank_account.deposit(50)
    assert bank_account.balance == 150
```

**Zalety fixture nad `setUp()`:**
- selektywne - tylko testy ktore potrzebuja, dostaja fixture,
- komponowalne - fixture moze zalezec od innej fixture,
- jeden test moze przyjac wiele fixtures naraz,
- IDE rozumie typ wstrzykiwanego obiektu (type hints).

| Cecha | `setUp` (unittest) | fixture (pytest) |
|-------|-------------------|------------------|
| Zasieg | cala klasa | wybrane testy |
| Kompozycja | trudna | naturalna (`f(a, b)`) |
| Teardown | `tearDown()` | `yield` w fixture |
| Wspoldzielenie | tylko w klasie | `conftest.py` |

> 💡 **Tip:** Kazde wywolanie testu dostaje swiezy egzemplarz
> fixture (domyslny scope `function`). Modyfikacje w jednym
> tescie nie wplywaja na kolejny - brak efektow ubocznych.

In [ ]:
import subprocess
import sys
import tempfile
import os


def _run(code, flags=None):
    """Zapisz kod jako plik testowy i uruchom przez pytest."""
    flags = flags or ['-v', '--tb=short', '-p', 'no:cacheprovider']
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='_test.py', delete=False, dir='.'
    ) as tmp:
        tmp.write(code)
        fname = tmp.name
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', fname] + flags,
        capture_output=True, text=True
    )
    short = os.path.basename(fname)
    print(result.stdout.replace(fname, short)[-1500:])
    os.unlink(fname)


_run('''
import pytest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        self.balance -= amount


# Fixture: swiezy obiekt przed kazdym testem
@pytest.fixture
def bank_account():
    return BankAccount(balance=100)


# Fixture zalezna od innej fixture (kompozycja)
@pytest.fixture
def funded_account():
    return BankAccount(balance=1000)


def test_deposit_increases_balance(bank_account):
    bank_account.deposit(50)
    assert bank_account.balance == 150


def test_each_test_gets_fresh_account(bank_account):
    # balance == 100, nie 70 z poprzedniego testu
    assert bank_account.balance == 100


def test_two_fixtures_in_one_test(bank_account, funded_account):
    bank_account.deposit(funded_account.balance)
    assert bank_account.balance == 1100
''')

---

### 🐍 Ćwiczenia - fixtures i wstrzykiwanie

**Ćw. 1.** Napisz fixture `bank_account` zwracajaca swiezy
obiekt `BankAccount(balance=100)`. Uzyj jej w 3 testach.

**Ćw. 2.** Napisz fixture `counter` dla klasy `Counter`.
Pokaz ze kazdy test dostaje izolowany egzemplarz -
modyfikacja w jednym tescie nie psuje kolejnego.

**Ćw. 4.** Napisz fixture `populated_cart` zalezna od
fixture `empty_cart`. Zalezna fixture przyjmuje inna fixture
jako parametr i dodaje do koszyka kilka elementow.

In [ ]:
# Ćw. 1: fixture bank_account - swiezy obiekt przed kazdym testem
_code = '''
import pytest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Amount must be positive')
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError('Insufficient funds')
        self.balance -= amount


# Uzupelnij fixture:
@pytest.fixture
def bank_account():
    ...  # hint: return BankAccount(balance=100)


def test_initial_balance_is_100(bank_account):
    assert ...  # hint: bank_account.balance == 100


def test_deposit_increases_balance(bank_account):
    bank_account.deposit(50)
    assert ...  # hint: bank_account.balance == 150


def test_withdraw_decreases_balance(bank_account):
    bank_account.withdraw(30)
    assert ...
'''
_run(_code)

In [ ]:
# Ćw. 2: ta sama fixture w 3 testach - izolacja egzemplarzow
_code = '''
import pytest


class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1

    def reset(self):
        self.value = 0


# Uzupelnij fixture:
@pytest.fixture
def counter():
    ...  # hint: return Counter()


def test_counter_starts_at_zero(counter):
    assert ...


def test_increment_once(counter):
    counter.increment()
    assert ...


def test_counter_is_fresh_after_previous_test(counter):
    # hint: wartosc powinna byc 0, nie 1 z poprzedniego testu
    assert counter.value == 0
'''
_run(_code)

In [ ]:
# Ćw. 4: fixture populated_cart zalezna od fixture empty_cart
_code = '''
import pytest


class ShoppingCart:
    def __init__(self):
        self.items = []

    def add_item(self, name, price):
        self.items.append({'name': name, 'price': price})

    def total(self):
        return sum(i['price'] for i in self.items)

    def is_empty(self):
        return len(self.items) == 0


# Fixture bazowa - pusty koszyk
@pytest.fixture
def empty_cart():
    ...  # hint: return ShoppingCart()


# Fixture zalezna - przyjmuje empty_cart jako parametr
@pytest.fixture
def populated_cart(empty_cart):
    ...  # hint: dodaj 2-3 produkty do empty_cart i go zwroc


def test_empty_cart_is_empty(empty_cart):
    assert ...


def test_populated_cart_has_items(populated_cart):
    assert not populated_cart.is_empty()


def test_populated_cart_total_is_positive(populated_cart):
    assert populated_cart.total() > 0
'''
_run(_code)

## 2. 🔹 Scope i cykl zycia

Parametr `scope` w `@pytest.fixture` kontroluje jak dlugo
egzemplarz fixture jest utrzymywany w pamieci:

| Scope | Kiedy tworzony | Kiedy niszczony | Typowe uzycie |
|-------|---------------|----------------|---------------|
| `function` | przed kazdym testem | po kazdym tescie | swiezy obiekt CUT |
| `class` | przed 1. testem klasy | po ostatnim klasy | wspolne dane klasy |
| `module` | przed 1. testem pliku | po ostatnim pliku | polaczenie DB |
| `session` | raz na sesje pytest | po wszystkich | zewnetrzny serwer |

```python
@pytest.fixture(scope='module')
def db_connection():
    conn = sqlite3.connect(':memory:')
    yield conn
    conn.close()   # teardown raz po ostatnim tescie pliku
```

**Fixture z parametrami** (`params`) - ten sam test uruchamia sie
wielokrotnie, raz dla kazdego parametru. `request.param`
dostarcza biezaca wartosc:

```python
@pytest.fixture(params=['sqlite', 'dict'])
def storage(request):
    if request.param == 'sqlite':
        return SqliteStorage()
    return DictStorage()

def test_store_item(storage):  # uruchamia sie 2 razy!
    storage.put('key', 'value')
    assert storage.get('key') == 'value'
```

> 💡 **Tip:** `scope='module'` to dobry wybor dla polaczen
> z baza danych w pamieci (`:memory:`). Drogie operacje setup
> wykonaja sie raz dla wszystkich testow w pliku.

In [ ]:
_run('''
import pytest
import sqlite3


# scope=module - jedno polaczenie dla wszystkich testow w pliku
@pytest.fixture(scope='module')
def db():
    conn = sqlite3.connect(':memory:')
    conn.execute('CREATE TABLE products (id INT, name TEXT, price REAL)')
    conn.executemany(
        'INSERT INTO products VALUES (?, ?, ?)',
        [(1, 'apple', 1.5), (2, 'banana', 0.8), (3, 'cherry', 3.0)]
    )
    conn.commit()
    yield conn       # testy uzywaja tego polaczenia
    conn.close()     # teardown: po ostatnim tescie w pliku


def test_product_count(db):
    count = db.execute('SELECT COUNT(*) FROM products').fetchone()[0]
    assert count == 3


def test_cheapest_product(db):
    row = db.execute(
        'SELECT name FROM products ORDER BY price LIMIT 1'
    ).fetchone()
    assert row[0] == 'banana'


# Fixture z params - test uruchamia sie raz dla kazdego param
@pytest.fixture(params=['asc', 'desc'])
def sort_order(request):
    return request.param


def test_sort_order_is_valid_string(sort_order):
    # Uruchamia sie 2x: sort_order='asc' i sort_order='desc'
    assert isinstance(sort_order, str)
    assert sort_order in ('asc', 'desc')
''')

---

### 🐍 Ćwiczenia - scope i cykl zycia

**Ćw. 3. *(Trudniejsze)*** Napisz fixture z `scope='module'`
dla polaczenia SQLite `:memory:`. Stworz tabele w fixture,
a w testach wykonuj zapytania. Sprawdz ze polaczenie jest
wspoldzielone miedzy testami.

**Ćw. 6. *(Trudniejsze)*** Napisz fixture z `params` dla
dwoch implementacji `Storage`: slownikowej i listowej.
Ten sam test powinien przejsc dla obu implementacji.

**Ćw. 12. *(Trudniejsze)*** Napisz fixture fabryczna
(`make_account`), ktora zwraca funkcje tworzaca `BankAccount`
z podanym saldem. Uzyj jej do tworzenia wielu kont w tescie.

In [ ]:
# Ćw. 3: fixture scope='module' dla polaczenia SQLite
_code = '''
import pytest
import sqlite3


# Uzupelnij fixture z scope='module':
@pytest.fixture(scope=...)
def db():
    # hint: conn = sqlite3.connect(':memory:')
    # hint: stworz tabele users (id INTEGER, name TEXT)
    # hint: yield conn
    # hint: conn.close()
    ...


def test_insert_and_count(db):
    # hint: INSERT INTO users i sprawdz COUNT(*)
    ...


def test_fetch_user_by_name(db):
    # hint: wstaw uzytkownika i pobierz po imieniu
    ...


def test_connection_is_shared(db):
    # scope=module: dane z poprzednich testow sa dostepne
    count = db.execute('SELECT COUNT(*) FROM users').fetchone()[0]
    assert count >= 1
'''
_run(_code)

In [ ]:
# Ćw. 6: fixture z params - dwie implementacje Storage
_code = '''
import pytest


class DictStorage:
    def __init__(self):        self._data = {}
    def put(self, key, val):   self._data[key] = val
    def get(self, key):        return self._data.get(key)
    def delete(self, key):     self._data.pop(key, None)
    def size(self):            return len(self._data)


class ListStorage:
    def __init__(self):        self._items = []
    def put(self, key, val):   self._items.append((key, val))
    def get(self, key):        return next((v for k,v in self._items if k==key), None)
    def delete(self, key):     self._items = [(k,v) for k,v in self._items if k!=key]
    def size(self):            return len(self._items)


# Uzupelnij fixture z params dla obu implementacji:
@pytest.fixture(params=[...])
def storage(request):
    # hint: params=['dict', 'list']
    # hint: if request.param == 'dict': return DictStorage()
    ...


# Ten sam test uruchamia sie 2x (raz dla kazdej implementacji):
def test_put_and_get(storage):
    storage.put('key', 'value')
    assert storage.get('key') == ...


def test_delete_item(storage):
    storage.put('x', 1)
    storage.delete('x')
    assert storage.get('x') is None


def test_empty_storage_size(storage):
    assert storage.size() == 0
'''
_run(_code)

In [ ]:
# Ćw. 12: fixture fabryczna - zwraca funkcje tworzaca obiekty
_code = '''
import pytest


class BankAccount:
    def __init__(self, balance=0, owner='anonymous'):
        self.balance = balance
        self.owner = owner

    def deposit(self, amount):
        self.balance += amount


# Uzupelnij fixture fabryczna:
@pytest.fixture
def make_account():
    # hint: zdefiniuj funkcje factory(balance=0, owner='anonymous')
    # hint: zwroc te funkcje
    ...


def test_create_account_with_custom_balance(make_account):
    # hint: account = make_account(balance=500)
    account = make_account(...)
    assert account.balance == 500


def test_create_multiple_independent_accounts(make_account):
    a1 = make_account(balance=100)
    a2 = make_account(balance=200)
    assert a1.balance != a2.balance


def test_create_account_with_owner(make_account):
    account = make_account(balance=100, owner='alice')
    assert account.owner == 'alice'
'''
_run(_code)

## 3. 🔹 Yield fixtures i `conftest.py`

**Yield fixtures** umozliwiaja teardown - czyszczenie zasobow
po tescie. Kod przed `yield` to setup, kod po `yield` teardown.
Teardown wykonuje sie nawet gdy test sie nie powiodl:

```python
@pytest.fixture
def temp_file():
    fd, path = tempfile.mkstemp(suffix='.txt')
    os.close(fd)
    yield path          # test dostaje sciezke do pliku
    os.unlink(path)     # teardown: usun plik po tescie
```

To odpowiada wzorcowi `try/finally`:
```python
# Rownowazne powyzszemu:
path = mkstemp()[1]
try:
    test_using(path)
finally:
    os.unlink(path)
```

**`conftest.py`** to specjalny plik rozpoznawany przez pytest.
Fixtures zdefiniowane w `conftest.py` sa dostepne dla wszystkich
testow w tym samym katalogu i podkatalogach - bez importu:

```
project/
  conftest.py          <- fixtures dostepne dla calosci projektu
  tests/
    conftest.py        <- fixtures dostepne tylko w tests/
    test_account.py    <- uzywa fixtures z obu conftest.py
```

> 💡 **Tip:** `yield` jest preferowany nad `request.addfinalizer()`
> - jest bardziej czytelny i naturalny. Kazdy `yield` fixture
> dziala jak context manager i zawsze uruchamia teardown.

In [ ]:
_run('''
import pytest
import tempfile
import os


# Yield fixture: setup -> yield -> teardown
@pytest.fixture
def temp_file():
    fd, path = tempfile.mkstemp(suffix='.txt')
    os.close(fd)
    print(f'  [setup]    stworz plik: {os.path.basename(path)}')
    yield path            # test dostaje sciezke
    print(f'  [teardown] usun plik: {os.path.basename(path)}')
    if os.path.exists(path):
        os.unlink(path)


def test_write_to_temp_file(temp_file):
    with open(temp_file, 'w', encoding='utf-8') as f:
        f.write('hello pytest fixtures')
    with open(temp_file, 'r', encoding='utf-8') as f:
        content = f.read()
    assert content == 'hello pytest fixtures'


def test_temp_file_exists_during_test(temp_file):
    assert os.path.exists(temp_file)


def test_temp_file_is_empty_initially(temp_file):
    # Kazdy test dostaje swiezy plik (scope=function)
    assert os.path.getsize(temp_file) == 0
''', flags=['-v', '--tb=short', '-s', '-p', 'no:cacheprovider'])

---

### 🐍 Ćwiczenia - yield fixtures i conftest.py

**Ćw. 5.** Napisz yield fixture `log_file` tworzaca tymczasowy
plik logu przed testem i usuwajaca go po tescie. Przetestuj
klase `Logger` zapisujaca wpisy do pliku.

**Ćw. 7.** Uzyj wbudowanej fixture `tmp_path` (obiekt
`pathlib.Path`) do testowania klasy `NoteStorage` zapisujacej
i odczytujacej pliki tekstowe.

**Ćw. 11.** Przeniés fixture `bank_account` do `conftest.py`
i sprawdz ze dziala w osobnym pliku testowym bez importu.

In [ ]:
# Ćw. 5: yield fixture tworzaca i usuwajaca plik logu
_code = '''
import pytest
import tempfile
import os


class Logger:
    def write(self, path, message):
        with open(path, 'a', encoding='utf-8') as f:
            f.write(message + '\n')

    def read_lines(self, path):
        with open(path, 'r', encoding='utf-8') as f:
            return f.readlines()


# Uzupelnij yield fixture:
@pytest.fixture
def log_file():
    # hint: fd, path = tempfile.mkstemp(suffix='.log')
    # hint: os.close(fd)
    # hint: yield path
    # hint: teardown: os.unlink(path)
    ...


def test_write_single_entry(log_file):
    logger = Logger()
    logger.write(log_file, 'INFO: started')
    lines = logger.read_lines(log_file)
    assert len(lines) == 1
    assert 'INFO' in lines[0]


def test_write_multiple_entries(log_file):
    logger = Logger()
    logger.write(log_file, 'DEBUG: step 1')
    logger.write(log_file, 'DEBUG: step 2')
    lines = logger.read_lines(log_file)
    assert len(lines) == ...


def test_log_file_is_fresh_per_test(log_file):
    # Kazdy test dostaje pusty plik (scope=function)
    logger = Logger()
    lines = logger.read_lines(log_file)
    assert lines == []
'''
_run(_code)

In [ ]:
# Ćw. 7: tmp_path - wbudowana fixture pytest
_code = '''
import pytest


class NoteStorage:
    def save(self, path, content):
        path.write_text(content, encoding='utf-8')

    def load(self, path):
        return path.read_text(encoding='utf-8')

    def exists(self, path):
        return path.exists()


# tmp_path to obiekt pathlib.Path - tymczasowy katalog
# pytest tworzy go i usuwa automatycznie

def test_save_creates_file(tmp_path):
    storage = NoteStorage()
    filepath = tmp_path / 'note.txt'
    # hint: storage.save(filepath, 'hello')
    # hint: assert storage.exists(filepath)
    ...


def test_load_returns_saved_content(tmp_path):
    storage = NoteStorage()
    filepath = tmp_path / 'note.txt'
    storage.save(filepath, 'test content')
    result = storage.load(filepath)
    assert result == ...


def test_multiple_files_in_tmp_dir(tmp_path):
    storage = NoteStorage()
    for i in range(3):
        storage.save(tmp_path / f'file_{i}.txt', f'content {i}')
    files = list(tmp_path.iterdir())
    assert len(files) == ...
'''
_run(_code)

In [ ]:
# Ćw. 11: conftest.py - fixtures wspoldzielone bez importu
import os
import shutil
import subprocess
import sys
import tempfile


# conftest.py - fixtures zdefiniowane tutaj dostepne w calym katalogu
_conftest = '''
import pytest


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Amount must be positive')
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError('Insufficient funds')
        self.balance -= amount


# Uzupelnij fixture w conftest.py:
@pytest.fixture
def bank_account():
    ...  # hint: return BankAccount(balance=100)
'''

# Plik testowy - NIE importuje fixtures, pytest sam je wstrzykuje
_test_file = '''
# Brak importu! Fixtures z conftest.py dostepne automatycznie.

def test_initial_balance(bank_account):
    assert bank_account.balance == 100


def test_deposit(bank_account):
    bank_account.deposit(50)
    assert bank_account.balance == 150


def test_withdraw(bank_account):
    bank_account.withdraw(30)
    assert bank_account.balance == 70
'''

tmpdir = tempfile.mkdtemp()
with open(os.path.join(tmpdir, 'conftest.py'), 'w') as f:
    f.write(_conftest)
with open(os.path.join(tmpdir, 'test_bank.py'), 'w') as f:
    f.write(_test_file)

result = subprocess.run(
    [sys.executable, '-m', 'pytest', tmpdir,
     '-v', '--tb=short', '-p', 'no:cacheprovider'],
    capture_output=True, text=True
)
print(result.stdout[-1200:])
shutil.rmtree(tmpdir)

## 4. 🔹 Wbudowane fixtures

pytest dostarcza kilka gotowych fixtures - nie wymagaja definicji
ani importu, wystarczy podac ich nazwe jako parametr:

**`tmp_path`** - tymczasowy katalog (`pathlib.Path`),
auto-czyszczony po sesji testowej.

**`capsys`** - przechwyt standardowego wyjscia i bledu:
```python
def test_print_output(capsys):
    print('hello')
    captured = capsys.readouterr()
    assert captured.out == 'hello\n'
```

**`monkeypatch`** - tymczasowe modyfikacje obiektow,
slownikow i zmiennych srodowiskowych (cofane po tescie):
```python
def test_env_var(monkeypatch):
    monkeypatch.setenv('API_URL', 'http://test')
    assert os.environ['API_URL'] == 'http://test'
# Po tescie: API_URL przywrocone do poprzedniej wartosci
```

| Fixture | Co robi |
|---------|---------|
| `tmp_path` | tymczasowy katalog `pathlib.Path` |
| `capsys` | przechwyt `sys.stdout` / `sys.stderr` |
| `monkeypatch` | podmiana atrybutow, env, importow |
| `request` | informacje o biezacym tescie/fixture |
| `capfd` | przechwyt niskiego poziomu (file descriptor) |

Metody `monkeypatch`:
- `setenv(name, value)` - ustaw zmienna srodowiskowa
- `delenv(name, raising=True)` - usun zmienna srodowiskowa
- `setattr(obj, name, value)` - podmien atrybut obiektu
- `setitem(mapping, name, value)` - podmien klucz w slowniku

> 💡 **Tip:** `monkeypatch` jest preferowany nad recznie
> ustawianiem `os.environ` w testach - automatycznie przywraca
> poprzednie wartosci nawet gdy test failuje.

In [ ]:
_run('''
import pytest
import os


# --- capsys ---
def notify(message, level='INFO'):
    print(f'[{level}] {message}')


def test_notify_prints_correct_format(capsys):
    notify('deploy complete')
    captured = capsys.readouterr()
    assert '[INFO] deploy complete' in captured.out


def test_notify_error_level(capsys):
    notify('disk full', level='ERROR')
    out, err = capsys.readouterr()
    assert '[ERROR]' in out


# --- monkeypatch ---
def get_db_url():
    return os.environ.get('DATABASE_URL', 'sqlite:///prod.db')


def test_db_url_from_env(monkeypatch):
    monkeypatch.setenv('DATABASE_URL', 'sqlite:///test.db')
    assert get_db_url() == 'sqlite:///test.db'


def test_db_url_default_when_env_absent(monkeypatch):
    monkeypatch.delenv('DATABASE_URL', raising=False)
    assert get_db_url() == 'sqlite:///prod.db'
''')

---

### 🐍 Ćwiczenia - wbudowane fixtures

**Ćw. 8.** Uzyj `capsys` do testowania funkcji `format_report()`
pisacej na stdout. Sprawdz zarowno zawartosc jak i format wyjscia.

**Ćw. 9.** Uzyj `monkeypatch.setenv` do podstawienia zmiennej
srodowiskowej `APP_ENV` i sprawdz ze `get_config()` zwraca
wlasciwe wartosci dla roznych srodowisk.

**Ćw. 10. *(Trudniejsze)*** Uzyj `monkeypatch.setattr` do
podmiany metody obiektu. Przetestuj `OrderService.place_order()`
podmieniajac `EmailSender.send()` na atrap (lambda).

In [ ]:
# Ćw. 8: capsys - przechwyt stdout
_code = '''
import pytest


def format_report(title, items):
    print(f'=== {title} ===')
    for i, item in enumerate(items, 1):
        print(f'  {i}. {item}')
    print(f'Total: {len(items)} items')


def test_report_contains_title(capsys):
    format_report('Sales', ['apple', 'banana'])
    captured = capsys.readouterr()
    # hint: assert 'Sales' in captured.out
    ...


def test_report_lists_all_items(capsys):
    items = ['alpha', 'beta', 'gamma']
    format_report('Inventory', items)
    captured = capsys.readouterr()
    # hint: sprawdz ze kazdy item jest w outpucie
    for item in items:
        assert ... in captured.out


def test_report_shows_total_count(capsys):
    format_report('Orders', ['x', 'y', 'z'])
    out, _ = capsys.readouterr()
    # hint: 'Total: 3 items' powinno byc w output
    ...
'''
_run(_code)

In [ ]:
# Ćw. 9: monkeypatch - podstawienie zmiennej srodowiskowej
_code = '''
import pytest
import os


def get_config():
    env = os.environ.get('APP_ENV', 'production')
    configs = {
        'production':  {'debug': False, 'db': 'prod.db'},
        'development': {'debug': True,  'db': 'dev.db'},
        'test':        {'debug': True,  'db': ':memory:'},
    }
    return configs.get(env, configs['production'])


def test_production_config_is_default(monkeypatch):
    monkeypatch.delenv('APP_ENV', raising=False)
    config = get_config()
    assert config['debug'] is False
    assert config['db'] == 'prod.db'


def test_test_config(monkeypatch):
    # hint: monkeypatch.setenv('APP_ENV', 'test')
    ...
    config = get_config()
    assert config['db'] == ':memory:'


def test_development_config(monkeypatch):
    ...
    config = get_config()
    assert config['debug'] is True
    assert config['db'] == 'dev.db'
'''
_run(_code)

In [ ]:
# Ćw. 10: monkeypatch.setattr - podmiana metody obiektu
_code = '''
import pytest


class EmailSender:
    def send(self, to, subject, body):
        # W produkcji wysyla prawdziwy email przez SMTP
        raise ConnectionError('Cannot connect to SMTP server')


class OrderService:
    def __init__(self):
        self.email_sender = EmailSender()
        self.confirmations_sent = []

    def send_confirmation(self, order_id, email):
        self.email_sender.send(
            to=email,
            subject=f'Order {order_id} confirmed',
            body=f'Your order {order_id} is confirmed.'
        )
        self.confirmations_sent.append(order_id)

    def place_order(self, order_id, email):
        self.send_confirmation(order_id, email)
        return {'id': order_id, 'status': 'confirmed'}


def test_place_order_returns_confirmed_status(monkeypatch):
    service = OrderService()
    # hint: monkeypatch.setattr(service.email_sender, 'send',
    #           lambda **kw: None)  lub lambda to, subject, body: None
    ...
    result = service.place_order('ORD-001', 'alice@example.com')
    assert result['status'] == 'confirmed'


def test_place_order_records_confirmation(monkeypatch):
    service = OrderService()
    # hint: podmieniamy send zeby nie rzucal wyjatku
    ...
    service.place_order('ORD-042', 'bob@example.com')
    assert 'ORD-042' in service.confirmations_sent
'''
_run(_code)